# Phase 1 One-Click Automated Pipeline (Kaggle GPU)

### **Why Kaggle GPU?**
- **30 hours/week** free GPU quota (GPU T4 x 2 or P100).
- **No Disconnection Issues**: Run headless via **"Save Version" -> "Save & Run All (Commit)"**.
  The notebook runs in the Kaggle cloud for up to 12 hours without needing your browser to stay open!
- **Pre-trained Adapter Included**: If `models/qlora_adapter` is bundled, training is automatically skipped and the pipeline jumps straight to Merge, GGUF Quantization, Evaluation, and Latency Benchmarking.

### **Required Notebook Settings (Right Sidebar)**
1. **Accelerator**: Select **GPU T4 x 2** (recommended) or **GPU P100**.
2. **Internet**: Toggle to **Internet on** (required for downloading the base Qwen model and pip packages).
3. **Persistence**: Default is fine.

In [ ]:
# Cell 0: GPU & Environment Verification
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available: {gpu_available}")

if not gpu_available:
    raise RuntimeError("ERROR: A CUDA-enabled GPU (e.g. T4 x 2 or P100) is required. In the right panel, select Settings -> Accelerator -> GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_count = torch.cuda.device_count()
print(f"Active GPU Device: {gpu_name} (Total devices: {gpu_count})")
!nvidia-smi


In [ ]:
# Cell 1: Setup Workspace & Copy/Unpack Bundle
import os
import sys
import zipfile
from pathlib import Path

proj_dir = Path('/kaggle/working/proj')
proj_dir.mkdir(parents=True, exist_ok=True)

# 1. Check if an uncompressed zip exists
zip_candidates = list(Path('/kaggle/input').rglob('*.zip')) + list(Path('/kaggle/working').glob('*.zip'))

if zip_candidates:
    bundle_path = zip_candidates[0]
    print(f"Found zip archive at {bundle_path}. Extracting to {proj_dir}...")
    with zipfile.ZipFile(bundle_path, 'r') as zf:
        zf.extractall(proj_dir)
else:
    # 2. Kaggle automatically extracts uploaded datasets into folders!
    found_dir = None
    for marker in ['tests', 'src', 'scripts']:
        for p in Path('/kaggle/input').rglob(marker):
            if p.is_dir():
                found_dir = p.parent
                break
        if found_dir:
            break

    if found_dir is None:
        dirs = [d for d in Path('/kaggle/input').iterdir() if d.is_dir()]
        if dirs:
            found_dir = dirs[0]
        else:
            raise FileNotFoundError("Could not find dataset in /kaggle/input. Please ensure t2s-kaggle-bundle is attached via '+ Add Input'.")

    print(f"Kaggle pre-extracted dataset detected at: {found_dir}. Copying to {proj_dir}...")
    os.system(f"cp -rf '{found_dir}'/* '{proj_dir}'/")

%cd /kaggle/working/proj
if str(proj_dir) not in sys.path:
    sys.path.insert(0, str(proj_dir))
print(f"Workspace successfully initialized at {proj_dir}.")
!ls -la


In [ ]:
# Cell 2: Install Pinned Dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets sqlglot psutil matplotlib llama-cpp-python gguf
!pip install -q -e /kaggle/working/proj
print("Dependencies installed successfully.")


In [ ]:
# Cell 3: Pre-flight Verification & Leakage Check
%cd /kaggle/working/proj
!python scripts/check_leakage.py
!pytest -q tests


In [ ]:
# Cell 4: Train QLoRA Model or Use Pre-Trained Adapter
import os
import sys
import json
import time
from pathlib import Path

proj_dir = Path('/kaggle/working/proj')
if proj_dir.exists():
    os.chdir(proj_dir)
if str(proj_dir) not in sys.path:
    sys.path.insert(0, str(proj_dir))

FORCE_RETRAIN = False
adapter_weights = Path("models/qlora_adapter/adapter_model.safetensors")

if adapter_weights.exists() and adapter_weights.stat().st_size > 10 * 1024 * 1024 and not FORCE_RETRAIN:
    size_mb = adapter_weights.stat().st_size / (1024 * 1024)
    print(f"[INFO] Pre-trained QLoRA adapter found ({size_mb:.2f} MB). Skipping training.")
else:
    print("[INFO] Starting QLoRA fine-tuning on Spider training split...")
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
        Trainer,
        TrainingArguments,
    )
    from src.t2s.train.train_qlora import collate_masked_prompt_completion, format_dataset_samples

    ckpt_dir = Path("/kaggle/working/checkpoints")
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    train_data = format_dataset_samples("data/processed/train.jsonl")
    val_data = format_dataset_samples("data/processed/val.jsonl")
    print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

    out_adapter_dir = Path("models/qlora_adapter")
    out_adapter_dir.mkdir(parents=True, exist_ok=True)

    t_args = TrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=2,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=30,
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        fp16=True,
        seed=42,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=t_args,
        train_dataset=train_data,
        eval_dataset=val_data,
        data_collator=lambda b: collate_masked_prompt_completion(b, tokenizer, 1024),
    )

    t0 = time.time()
    trainer.train()
    train_time_sec = round(time.time() - t0, 2)

    trainer.save_model(str(out_adapter_dir))
    tokenizer.save_pretrained(str(out_adapter_dir))
    print(f"Saved adapter to {out_adapter_dir}")

    results_train_dir = Path("results/training")
    results_train_dir.mkdir(parents=True, exist_ok=True)
    training_info = {
        "gpu": torch.cuda.get_device_name(0),
        "training_time_sec": train_time_sec,
        "epochs": 2,
        "seed": 42,
        "train_examples": len(train_data),
        "val_examples": len(val_data),
        "final_loss": trainer.state.log_history[-1].get("loss", 0.0) if trainer.state.log_history else 0.0,
    }
    with open(results_train_dir / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(training_info, f, indent=2)


In [ ]:
# Cell 5: Merge QLoRA Adapter Into Base Weights (FP16)
!python scripts/merge_adapter.py --base-model Qwen/Qwen2.5-Coder-3B-Instruct --adapter-dir models/qlora_adapter --output-dir models/merged_model


In [ ]:
# Cell 6: Build llama.cpp and Export GGUF Q4_K_M
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /tmp/llama.cpp
%cd /tmp/llama.cpp
!mkdir -p build && cd build && cmake .. -DLLAMA_BUILD_EXAMPLES=ON -DLLAMA_BUILD_TESTS=OFF && cmake --build . --config Release --target llama-quantize -j 4
%cd /kaggle/working/proj
!python /tmp/llama.cpp/convert_hf_to_gguf.py models/merged_model --outtype f16 --outfile models/merged_f16.gguf
!/tmp/llama.cpp/build/bin/llama-quantize models/merged_f16.gguf models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf Q4_K_M

import os
if os.path.exists('models/merged_f16.gguf'):
    os.remove('models/merged_f16.gguf')
    print('Removed temporary FP16 GGUF file to free disk space.')

# Verify loading via llama_cpp
from llama_cpp import Llama
llm = Llama(model_path='models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf', n_gpu_layers=-1, verbose=False)
out = llm('SELECT', max_tokens=1)
print('GGUF Test Generation Token:', out['choices'][0]['text'])


In [ ]:
# Cell 7: Register Model Artifacts
!python scripts/register_artifacts.py


In [ ]:
# Cell 8: Evaluate Configs A, B, C (Transformers GPU Backend)
!python scripts/eval_spider.py --backend transformers --model-path Qwen/Qwen2.5-Coder-3B-Instruct --config A
!python scripts/eval_spider.py --backend transformers --model-path Qwen/Qwen2.5-Coder-3B-Instruct --config B
!python scripts/eval_spider.py --backend transformers --model-path models/merged_model --config C


In [ ]:
# Cell 9: Evaluate Config D (Quantized GGUF Q4_K_M)
!python scripts/eval_spider.py --backend llama_cpp --model-path models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf --config D


In [ ]:
# Cell 10: Benchmark Latency & Memory (Config D)
!python scripts/bench_latency.py --backend llama_cpp --model-path models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf --limit 200


In [ ]:
# Cell 11: Merge Results, Generate Figures & Run Verification Checks
!python scripts/merge_eval_results.py
!python scripts/make_figures.py
!python scripts/make_eval_doc.py
!python scripts/check_phase1.py


In [ ]:
# Cell 12: Bundle Results to /kaggle/working/phase1_results.zip
import zipfile
from pathlib import Path

out_zip = Path('/kaggle/working/phase1_results.zip')
if out_zip.exists():
    out_zip.unlink()

print('Bundling phase1_results.zip to /kaggle/working/...')
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in Path('results').rglob('*'):
        if p.is_file() and 'smoke' not in p.parts:
            zf.write(p, arcname=p.as_posix())
    for p in Path('models/qlora_adapter').rglob('*'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())
    for p in Path('models').glob('*.gguf'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())

size_mb = out_zip.stat().st_size / (1024 * 1024)
print(f'Successfully created phase1_results.zip ({size_mb:.2f} MB) in /kaggle/working/.')
print('You can download phase1_results.zip from the Output tab on Kaggle, then run locally:')
print('  python scripts/ingest_colab_results.py --zip path/to/phase1_results.zip')
